# AI Customer Feedback Intelligence System
### Graduation Project — Project 1


## 1. Setup

In [1]:
import re
import json
import pickle
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report,
                              silhouette_score)
from scipy.sparse import hstack, csr_matrix

sns.set_theme(style="whitegrid")


## 2. Problem Understanding & Dataset Exploration

In [2]:
df = pd.read_csv("/content/reviews_raw.csv")
print(df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/reviews_raw.csv'

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df["Rating"].value_counts().sort_index()

## 3. Data Cleaning & Preparation

In [ ]:
print("Duplicates to be removed:",df.duplicated().sum())
df = df.drop_duplicates()
df.dropna(subset=["Review Text"],inplace =True )
df["Division Name"] = df["Division Name"].fillna("Unknown")
print("Shape after cleaning:", df.shape)

In [ ]:
STOPWORDS = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 2]
    return " ".join(tokens)

df["clean_text"] = df["Review Text"].apply(clean_text)
df = df[df["clean_text"].str.len() > 0]

def sentiment_label(r):
    if r >= 4: return "Positive"
    if r == 3: return "Neutral"
    return "Negative"

df["Sentiment"] = df["Rating"].apply(sentiment_label)
df[["Review Text", "clean_text", "Rating", "Sentiment"]].head()

## 4. Feature Engineering

In [ ]:
df["review_length"] = df["Review Text"].str.len()
df["word_count"] = df["clean_text"].apply(lambda x: len(x.split()))
df["exclaim_count"] = df["Review Text"].str.count("!")

POS_LEX = {"love","good","great","perfect","amazing","excellent","beautiful",
           "comfortable","recommend","favorite","soft","stylish"}
NEG_LEX = {"disappointed","cheap","poor","returning","return","small",
           "tight","thin","damaged","late","expensive","overpriced"}

df["pos_lex_count"] = df["clean_text"].apply(lambda x: sum(w in POS_LEX for w in x.split()))
df["neg_lex_count"] = df["clean_text"].apply(lambda x: sum(w in NEG_LEX for w in x.split()))
df[["review_length","word_count","exclaim_count","pos_lex_count","neg_lex_count"]].describe()

## 5. Exploratory Data Analysis / Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
sns.countplot(data=df, x="Sentiment", hue="Sentiment", legend=False,
              order=["Positive","Neutral","Negative"],
              palette=["#4C9A6A","#E8B84B","#D1495B"], ax=ax)
ax.set_title("Customer Sentiment Distribution")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
sns.countplot(data=df, x="Rating", order=[1,2,3,4,5], color="#4472C4", ax=ax)
ax.set_title("Review Rating Distribution")
plt.tight_layout(); plt.show()

In [ ]:
all_words = " ".join(df["clean_text"]).split()
common = Counter(all_words).most_common(15)
words, counts = zip(*common)
fig, ax = plt.subplots(figsize=(7,5))
sns.barplot(x=list(counts), y=list(words), color="#6A5ACD", ax=ax)
ax.set_title("Top 15 Most Common Words in Reviews")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
sns.countplot(data=df, x="Department Name", hue="Department Name", legend=False,
              order=df["Department Name"].value_counts().index, color="#DD8452", ax=ax)
ax.set_title("Reviews by Department")
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

## 6. NLP Processing — TF-IDF Vectorization


In [ ]:
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2), min_df=3)
X_text = tfidf.fit_transform(df["clean_text"])

numeric_feats = df[["review_length","word_count","exclaim_count",
                     "pos_lex_count","neg_lex_count"]].values
X_full = hstack([X_text, csr_matrix(numeric_feats)])
y = df["Sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

## 7. Machine Learning — Classification Models


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=250, random_state=42, class_weight="balanced"),
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(class_weight="balanced", max_iter=5000),
}

model_results, trained_models = {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    model_results[name] = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, average="weighted"),
        "recall": recall_score(y_test, preds, average="weighted"),
        "f1": f1_score(y_test, preds, average="weighted"),
    }
    trained_models[name] = model

pd.DataFrame(model_results).T.sort_values("f1", ascending=False)

In [ ]:
comp_df = pd.DataFrame(model_results).T.reset_index().rename(columns={"index":"Model"})
comp_melt = comp_df.melt(id_vars="Model", var_name="Metric", value_name="Score")
fig, ax = plt.subplots(figsize=(8,5))
sns.barplot(data=comp_melt, x="Model", y="Score", hue="Metric", ax=ax)
ax.set_title("Model Performance Comparison"); ax.set_ylim(0,1)
plt.xticks(rotation=15); plt.tight_layout(); plt.show()

## 8. Model Evaluation

In [ ]:
best_name = max(model_results, key=lambda k: model_results[k]["f1"])
best_model = trained_models[best_name]
print("Best model:", best_name)

preds_best = best_model.predict(X_test)
print(classification_report(y_test, preds_best))

In [ ]:
labels_order = ["Positive","Neutral","Negative"]
cm = confusion_matrix(y_test, preds_best, labels=labels_order)
fig, ax = plt.subplots(figsize=(5.5,4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels_order, yticklabels=labels_order, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix — {best_name}")
plt.tight_layout(); plt.show()

## 9. Unsupervised Learning — K-Means Complaint Clustering


In [ ]:
neg_df = df[df["Sentiment"] == "Negative"].copy()
neg_tfidf = TfidfVectorizer(max_features=1500, ngram_range=(1,2), min_df=2)
X_neg = neg_tfidf.fit_transform(neg_df["clean_text"])

k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
neg_df["cluster"] = kmeans.fit_predict(X_neg)

sil = silhouette_score(X_neg, neg_df["cluster"])
print("Silhouette score:", round(sil, 3))

terms = neg_tfidf.get_feature_names_out()
for c in range(k):
    top_idx = kmeans.cluster_centers_[c].argsort()[::-1][:6]
    print(f"Cluster {c}: {[terms[i] for i in top_idx]}")

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_neg.toarray())
fig, ax = plt.subplots(figsize=(7,5.5))
scatter = ax.scatter(coords[:,0], coords[:,1], c=neg_df["cluster"], cmap="tab10", s=25, alpha=0.75)
ax.set_title("Customer Complaint Clusters (PCA projection)")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.add_artist(ax.legend(*scatter.legend_elements(), title="Cluster"))
plt.tight_layout(); plt.show()

In [ ]:
cluster_sizes = neg_df["cluster"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6,4))
sns.barplot(x=cluster_sizes.index.astype(str), y=cluster_sizes.values, color="#B85450", ax=ax)
ax.set_title("Negative Reviews per Complaint Cluster")
ax.set_xlabel("Cluster"); ax.set_ylabel("Count")
plt.tight_layout(); plt.show()

## 10. Business Insights

In [ ]:
insights = {
    "overall_positive_rate": (df["Sentiment"]=="Positive").mean(),
    "overall_negative_rate": (df["Sentiment"]=="Negative").mean(),
    "recommend_rate": df["Recommended IND"].mean(),
    "worst_department": df.groupby("Department Name")["Rating"].mean().idxmin(),
    "best_department": df.groupby("Department Name")["Rating"].mean().idxmax(),
    "avg_rating_by_department": df.groupby("Department Name")["Rating"].mean().round(2).to_dict(),
}
insights

## 11. Save Artifacts for Deployment

In [ ]:
import os
os.makedirs("models", exist_ok=True)

with open("models/sentiment_model.pkl", "wb") as f:
    pickle.dump(best_model, f)
with open("models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)
with open("models/complaint_kmeans.pkl", "wb") as f:
    pickle.dump(kmeans, f)
with open("models/complaint_tfidf.pkl", "wb") as f:
    pickle.dump(neg_tfidf, f)

cluster_labels = {
    str(c): [terms[i] for i in kmeans.cluster_centers_[c].argsort()[::-1][:6]]
    for c in range(k)
}
with open("models/cluster_labels.json", "w") as f:
    json.dump(cluster_labels, f, indent=2)

print("Artifacts saved to models/")

## 12. Deployment — Gradio App

In [ ]:
# Quick smoke-test of the saved model (same logic app.py uses)
def predict_sentiment(text):
    clean = clean_text(text)
    vec_text = tfidf.transform([clean])
    length, wc, exc = len(text), len(clean.split()), text.count("!")
    pos_c = sum(w in POS_LEX for w in clean.split())
    neg_c = sum(w in NEG_LEX for w in clean.split())
    vec_full = hstack([vec_text, csr_matrix([[length, wc, exc, pos_c, neg_c]])])
    pred = best_model.predict(vec_full)[0]
    return pred

for sample in ["Absolutely love this dress, perfect fit!",
               "The fabric was cheap and it arrived late, very disappointed.",
               "It's an okay top, nothing special."]:
    print(sample, "->", predict_sentiment(sample))

In [ ]:
import re
import json
import pickle
from pathlib import Path

import gradio as gr
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

BASE = Path.cwd()
STOPWORDS = set(ENGLISH_STOP_WORDS)

POS_LEX = {"love", "great", "perfect", "amazing", "excellent", "beautiful",
           "comfortable", "recommend", "favorite", "soft", "stylish"}
NEG_LEX = {"disappointed", "cheap", "poor", "returning", "return", "small",
           "tight", "thin", "damaged", "late", "expensive", "overpriced"}


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = [t for t in text.split() if t not in STOPWORDS and len(t) > 2]
    return " ".join(tokens)


# ---------------------------------------------------------------
# Load trained artifacts
# ---------------------------------------------------------------
with open(BASE / "models/sentiment_model.pkl", "rb") as f:
    sentiment_model = pickle.load(f)
with open(BASE / "models/tfidf_vectorizer.pkl", "rb") as f:
    tfidf = pickle.load(f)
with open(BASE / "models/complaint_kmeans.pkl", "rb") as f:
    kmeans = pickle.load(f)
with open(BASE / "models/complaint_tfidf.pkl", "rb") as f:
    complaint_tfidf = pickle.load(f)
with open(BASE / "models/cluster_labels.json") as f:
    cluster_labels = json.load(f)


def build_features(raw_text):
    clean = clean_text(raw_text)
    vec_text = tfidf.transform([clean])
    length, wc, exc = len(raw_text), len(clean.split()), raw_text.count("!")
    pos_c = sum(w in POS_LEX for w in clean.split())
    neg_c = sum(w in NEG_LEX for w in clean.split())
    numeric = csr_matrix([[length, wc, exc, pos_c, neg_c]])
    return hstack([vec_text, numeric]), clean


# ---------------------------------------------------------------
# Tab 1: Sentiment classifier
# ---------------------------------------------------------------
def predict_sentiment(review_text):
    if not review_text or not review_text.strip():
        return "—", {}

    X, clean = build_features(review_text)
    pred = sentiment_model.predict(X)[0]

    scores = {}
    if hasattr(sentiment_model, "predict_proba"):
        proba = sentiment_model.predict_proba(X)[0]
        classes = sentiment_model.classes_
        scores = {c: float(p) for c, p in zip(classes, proba)}
    elif hasattr(sentiment_model, "decision_function"):
        # Linear SVM: convert decision scores to a pseudo-confidence via softmax
        import numpy as np
        d = sentiment_model.decision_function(X)[0]
        d = np.atleast_1d(d)
        exp = np.exp(d - d.max())
        soft = exp / exp.sum()
        classes = sentiment_model.classes_
        scores = {c: float(p) for c, p in zip(classes, soft)}

    cluster_note = ""
    if pred == "Negative":
        cvec = complaint_tfidf.transform([clean])
        cluster_id = str(kmeans.predict(cvec)[0])
        terms = ", ".join(cluster_labels.get(cluster_id, [])[:5])
        cluster_note = f"Likely complaint theme: **{terms}**"

    label = f"### Predicted sentiment: **{pred}**"
    if cluster_note:
        label += f"\n\n{cluster_note}"

    return label, scores


with gr.Blocks(title="AI Customer Feedback Intelligence System") as demo:
    gr.Markdown("# 🛍️ AI Customer Feedback Intelligence System")
    gr.Markdown(
        "Graduation Project — Project 1 · Sentiment classification, complaint clustering "
        "and a business insights dashboard for e-commerce clothing reviews."
    )

    with gr.Tab("Sentiment Classifier"):
        gr.Markdown("Paste a customer review and get an instant sentiment prediction.")
        with gr.Row():
            with gr.Column():
                inp = gr.Textbox(
                    label="Customer review",
                    lines=5,
                    placeholder="e.g. The fabric felt cheap and it arrived a week late...",
                )
                btn = gr.Button("Analyze Sentiment", variant="primary")
                gr.Examples(
                    examples=[
                        "Absolutely love this dress! Perfect fit and gorgeous color.",
                        "The fabric felt cheap and it started pilling after one wash.",
                        "It's an okay top, nothing special but wearable.",
                        "Runs way too small, had to return it immediately.",
                    ],
                    inputs=inp,
                )
            with gr.Column():
                out_label = gr.Markdown()
                out_scores = gr.Label(label="Confidence by class")
        btn.click(predict_sentiment, inputs=inp, outputs=[out_label, out_scores])
        inp.submit(predict_sentiment, inputs=inp, outputs=[out_label, out_scores])

    with gr.Tab("Complaint Cluster Explorer"):
        gr.Markdown(
            "The K-Means model groups **negative reviews** into recurring complaint themes. "
            "Here's what each cluster is about, based on its top TF-IDF terms:"
        )
        rows = [[f"Cluster {cid}", ", ".join(terms)] for cid, terms in cluster_labels.items()]
        gr.Dataframe(
            headers=["Cluster", "Top terms"],
            value=rows,
            interactive=False,
        )
        gr.Image(str(BASE / "charts/complaint_clusters.png"), label="Cluster visualization (PCA)")
        gr.Image(str(BASE / "charts/cluster_sizes.png"), label="Reviews per cluster")

    with gr.Tab("Insights Dashboard"):
        gr.Markdown("### Business Insights Dashboard")
        with gr.Row():
            gr.Image(str(BASE / "charts/sentiment_distribution.png"))
            gr.Image(str(BASE / "charts/rating_distribution.png"))
        with gr.Row():
            gr.Image(str(BASE / "charts/common_words.png"))
            gr.Image(str(BASE / "charts/department_distribution.png"))
        with gr.Row():
            gr.Image(str(BASE / "charts/model_comparison.png"))
            gr.Image(str(BASE / "charts/confusion_matrix.png"))
demo.launch()  # add share=True for a public link